### Delete the abnormal data:


- Sudden drops in pressure and abrupt increases in flow are likely due to suction.



### Abnormal Detection

- Go through all the CSV files. If the 'CDGR - PEEP' / 'AVEA - PEEP' / 'SVU - PEEP' is less than or equal to 80% of the PEEP setting, then delete the corresponding rows.



Delete the abnormal data:

- If there are PEEP settings, the abnormal data define as when the actual PEEP values lower than the 80% of the respective PEEP settings.

- If the PEEP settings are absent, then calculate the 1 min (or 5 min) moving average of actual PEEP values, then the abonormal data defines as lower than the 80% of the respective moving average of the actual PEEP values. 

Note: when deleting, delete the corresponding rows include other variables such as PIP, eVT, etc. We use PEEP as a signal to detect abnormal events, such as suction event, etc.

In [1]:
### Calculate 80% of the PEEP Settings or 5-min moving average of the actual PEEP values
### PARDS Risk V3 1st-hour
import os
import pandas as pd

# Input directory (read-only here)
input_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED"

# Output directory (writable)
output_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st"
os.makedirs(output_dir, exist_ok=True)

# PEEP setting and actual PEEP column candidates
peep_setting_cols = [
    "CAPSULE_AVEAA_VITAL_2584",       # AVEA - Set: PEEP
    "CAPSULE_DRAGERMEDIBUS_VITAL_2104",  # CDGR - Set: PEEP L
    "CAPSULE_MAQUETC_VITAL_2104"      # SVU - Set: PEEP L
]

actual_peep_cols = [
    "CAPSULE_AVEAA_VITAL_1189",       # AVEA - PEEP
    "CAPSULE_DRAGERMEDIBUS_VITAL_1189",  # CDGR - PEEP
    "CAPSULE_MAQUETC_VITAL_1189"      # SVU - PEEP
]

# Loop through all CSV files
for filename in os.listdir(input_dir):
    if filename.endswith(".csv"):
        filepath = os.path.join(input_dir, filename)
        df = pd.read_csv(filepath)

        peep_setting_series = None
        actual_peep_series = None

        # Check for usable PEEP setting column
        for col in peep_setting_cols:
            if col in df.columns and not df[col].dropna().empty:
                peep_setting_series = df[col]
                break

        if peep_setting_series is not None:
            df["PEEP_setting"] = peep_setting_series
            df["PEEP_setting_80pct"] = 0.8 * peep_setting_series
        else:
            # Fallback: find usable actual PEEP column
            for col in actual_peep_cols:
                if col in df.columns and not df[col].dropna().empty:
                    actual_peep_series = df[col]
                    break
            if actual_peep_series is not None:
                df["PEEP_actual"] = actual_peep_series
                df["PEEP_actual_MA_5min"] = actual_peep_series.rolling(window=300, min_periods=1).mean()

        # Save to new location
        df.to_csv(os.path.join(output_dir, filename), index=False)

print(f"Processed files saved to: {output_dir}")


Processed files saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st


In [2]:
### Plot images BEFORE abnormal data deletion
### PARDS Risk V3 1st-hour
import os
import pandas as pd
import matplotlib.pyplot as plt

# === INPUT/OUTPUT PATHS ===
csv_directory = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st"
images_directory = os.path.join(csv_directory, "images_before_deletion_V3_1st")
os.makedirs(images_directory, exist_ok=True)

# === PEEP columns and unit conversion ===
actual_peep_candidates = {
    "CAPSULE_AVEAA_VITAL_1189": 1.0,       # AVEA - PEEP (already in cmH₂O)
    "CAPSULE_MAQUETC_VITAL_1189": 1.0,     # SVU - PEEP (already in cmH₂O)
    "CAPSULE_DRAGERMEDIBUS_VITAL_1189": 1.01972  # CDGR - PEEP (convert mbar → cmH₂O)
}

# === Process CSVs ===
for filename in os.listdir(csv_directory):
    if not filename.endswith(".csv"):
        continue

    filepath = os.path.join(csv_directory, filename)
    df = pd.read_csv(filepath)

    # Identify actual PEEP column with data
    actual_col = None
    conversion_factor = 1.0
    for col, factor in actual_peep_candidates.items():
        if col in df.columns and not df[col].dropna().empty:
            actual_col = col
            conversion_factor = factor
            break

    if actual_col is None or ("PEEP_setting" not in df.columns and "PEEP_actual_MA_5min" not in df.columns):
        print(f"Skipping {filename}: No usable PEEP column found.")
        continue

    # Use PEEP setting or fallback to moving average
    peep_setting_col = "PEEP_setting" if "PEEP_setting" in df.columns else "PEEP_actual_MA_5min"

    # Convert Time column
    if "Time" not in df.columns:
        print(f"Skipping {filename}: No 'Time' column.")
        continue
    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
    df = df.dropna(subset=["Time"])

    # Convert actual PEEP to cmH₂O if needed
    df["Actual_PEEP_cmH2O"] = df[actual_col] * conversion_factor

    # === Plot ===
    plt.figure(figsize=(16, 8))
    plt.plot(df["Time"], df["Actual_PEEP_cmH2O"], label="Actual PEEP (cmH₂O)", color="blue")

    # Add PEEP setting and 80% threshold lines
    first_solid = first_dashed = True
    for setting_val in df[peep_setting_col].dropna().unique():
        times = df[df[peep_setting_col] == setting_val]["Time"]
        if times.empty:
            continue

        # Solid line (setting)
        plt.hlines(y=setting_val, xmin=times.min(), xmax=times.max(),
                   colors="red", linestyles="-", linewidth=3,
                   label="PEEP Setting" if first_solid else None)
        first_solid = False

        # Dashed line (80% of setting)
        plt.hlines(y=0.8 * setting_val, xmin=times.min(), xmax=times.max(),
                   colors="red", linestyles="--", linewidth=3,
                   label="80% of PEEP Setting" if first_dashed else None)
        first_dashed = False

    # === Format plot ===
    plt.xlabel("Time", fontsize=24)
    plt.ylabel("PEEP Value ($cmH_2O$)", fontsize=24)
    plt.title("PEEP Values and Settings Before Abnormal Data Removal", fontsize=28)
    plt.tick_params(axis='both', labelsize=20)
    # plt.ylim(0, 15)
    plt.legend(loc='upper center', bbox_to_anchor=(1.15, 1.0), fontsize=18)
    plt.tight_layout()

    # Save
    base = os.path.join(images_directory, filename.replace(".csv", ""))
    plt.savefig(f"{base}.pdf", format="pdf", bbox_inches="tight")
    plt.close()

print("✅ All plots saved!")


Skipping 373213_20240918_18_1st_Raw.csv: No usable PEEP column found.
Skipping 510530_20240916_17_1st_Raw.csv: No usable PEEP column found.
Skipping 84839_20240918_16_1st_Raw.csv: No usable PEEP column found.
✅ All plots saved!


In [3]:
### Abnormal data deletion
### PARDS Risk V3 1st-hour
import os
import pandas as pd

# === Paths ===
csv_directory = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st"
destination_directory = os.path.join(csv_directory, "Abnormal_Deletion_V3_1st")
os.makedirs(destination_directory, exist_ok=True)

# === Actual PEEP columns with ventilator mapping ===
actual_peep_columns = [
    "CAPSULE_AVEAA_VITAL_1189",        # AVEA - PEEP
    "CAPSULE_DRAGERMEDIBUS_VITAL_1189", # CDGR - PEEP
    "CAPSULE_MAQUETC_VITAL_1189"       # SVU - PEEP
]

# === Loop through all files ===
for filename in os.listdir(csv_directory):
    if not filename.endswith(".csv"):
        continue

    filepath = os.path.join(csv_directory, filename)
    df = pd.read_csv(filepath)

    # Ensure usable actual PEEP column exists
    actual_col = next((col for col in actual_peep_columns if col in df.columns and not df[col].dropna().empty), None)
    if actual_col is None:
        print(f"⚠️ Skipping {filename}: No usable actual PEEP column.")
        continue

    # Ensure 'PEEP_setting_80pct' column is present and valid
    if "PEEP_setting_80pct" not in df.columns or df["PEEP_setting_80pct"].dropna().empty:
        print(f"⚠️ Skipping {filename}: 'PEEP_setting_80pct' column missing or empty.")
        continue

    # Ensure Time column is valid
    if "Time" not in df.columns:
        print(f"⚠️ Skipping {filename}: No 'Time' column.")
        continue

    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
    df = df.dropna(subset=["Time"])
    df = df.set_index("Time")

    # Remove rows where actual PEEP < 80% of PEEP setting
    df = df[~(df[actual_col] < df["PEEP_setting_80pct"])]

    # Save cleaned file
    df.reset_index(inplace=True)
    cleaned_path = os.path.join(destination_directory, filename)
    df.to_csv(cleaned_path, index=False)

print("✅ Abnormal data deletion completed using 'PEEP_setting_80pct'.")
print(f"Cleaned files saved to: {destination_directory}")



⚠️ Skipping 373213_20240918_18_1st_Raw.csv: No usable actual PEEP column.
⚠️ Skipping 510530_20240916_17_1st_Raw.csv: No usable actual PEEP column.
⚠️ Skipping 84839_20240918_16_1st_Raw.csv: No usable actual PEEP column.
✅ Abnormal data deletion completed using 'PEEP_setting_80pct'.
Cleaned files saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st/Abnormal_Deletion_V3_1st


In [4]:
### Plot images AFTER abnormal data deletion
### PARDS Risk V3 1st-hour
import os
import pandas as pd
import matplotlib.pyplot as plt

# === INPUT/OUTPUT PATHS ===
csv_directory = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st/Abnormal_Deletion_V3_1st"
images_directory = os.path.join(csv_directory, "images_after_deletion_V3_1st")
os.makedirs(images_directory, exist_ok=True)

# === Actual PEEP columns and unit conversion ===
actual_peep_candidates = {
    "CAPSULE_AVEAA_VITAL_1189": 1.0,       # AVEA - PEEP (cmH₂O)
    "CAPSULE_MAQUETC_VITAL_1189": 1.0,     # SVU - PEEP (cmH₂O)
    "CAPSULE_DRAGERMEDIBUS_VITAL_1189": 1.01972  # CDGR - PEEP (mbar → cmH₂O)
}

# === Loop through all CSV files ===
for filename in os.listdir(csv_directory):
    if not filename.endswith(".csv"):
        continue

    filepath = os.path.join(csv_directory, filename)
    df = pd.read_csv(filepath)

    # Identify the actual PEEP column with data
    actual_col = None
    conversion_factor = 1.0
    for col, factor in actual_peep_candidates.items():
        if col in df.columns and not df[col].dropna().empty:
            actual_col = col
            conversion_factor = factor
            break

    if actual_col is None:
        print(f"Skipping {filename}: No actual PEEP data found.")
        continue

    # Use PEEP_setting or fallback to PEEP_actual_MA_5min
    if "PEEP_setting" in df.columns and not df["PEEP_setting"].dropna().empty:
        peep_setting_col = "PEEP_setting"
    elif "PEEP_actual_MA_5min" in df.columns and not df["PEEP_actual_MA_5min"].dropna().empty:
        peep_setting_col = "PEEP_actual_MA_5min"
    else:
        print(f"Skipping {filename}: No usable setting or moving average column.")
        continue

    # Parse and clean time
    if "Time" not in df.columns:
        print(f"Skipping {filename}: No 'Time' column.")
        continue
    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
    df = df.dropna(subset=["Time"])

    # Convert actual PEEP to cmH₂O
    df["Actual_PEEP_cmH2O"] = df[actual_col] * conversion_factor

    # === Plot ===
    plt.figure(figsize=(16, 8))
    plt.plot(df["Time"], df["Actual_PEEP_cmH2O"], label="Actual PEEP (cmH₂O)", color="blue")

    first_solid = first_dashed = True
    for setting_val in df[peep_setting_col].dropna().unique():
        times = df[df[peep_setting_col] == setting_val]["Time"]
        if times.empty:
            continue

        plt.hlines(y=setting_val, xmin=times.min(), xmax=times.max(),
                   colors="red", linestyles="-", linewidth=3,
                   label="PEEP Setting" if first_solid else None)
        first_solid = False

        plt.hlines(y=0.8 * setting_val, xmin=times.min(), xmax=times.max(),
                   colors="red", linestyles="--", linewidth=3,
                   label="80% of PEEP Setting" if first_dashed else None)
        first_dashed = False

    # Finalize plot
    plt.xlabel("Time", fontsize=24)
    plt.ylabel("PEEP Value ($cmH_2O$)", fontsize=24)
    plt.title("PEEP Values and Settings After Abnormal Data Removal", fontsize=28)
    plt.tick_params(axis='both', labelsize=20)
    plt.ylim(0, 15)
    plt.legend(loc='upper center', bbox_to_anchor=(1.15, 1.0), fontsize=18)
    plt.tight_layout()

    # Save
    base_name = os.path.join(images_directory, filename.replace(".csv", ""))
    plt.savefig(f"{base_name}.pdf", format="pdf", bbox_inches="tight")
    plt.close()

print("✅ All plots saved to:", images_directory)


✅ All plots saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V3/V3_1st_Raw/Study23_Tag282_EventList/RENAMED/PEEP_Settings_V3_1st/Abnormal_Deletion_V3_1st/images_after_deletion_V3_1st
